# Inference -- any fundus image -> every stage up to PDR

Give it any fundus photo (or a folder of them). Module 1 grades it and segments its lesions,
then Module 2 cascades stage by stage up to stage 4 (PDR): generate the next stage -> re-segment
it with Module 1 -> re-grade it (consistency check) -> feed it forward. Optionally Module 3
captions each panel with the estimated chance of further progression within 2 years.

Unlike notebook 08 (Tianjin patients only, stopping at their real follow-up grade), this needs no
dataset or follow-up image. Wraps `generate_progression.py`.

**Needs:** Module 1 checkpoints (notebooks 02/03) and a Module 2 Generator -- notebook 06 now
copies it to `Thesis_Datasets/module2_runs/<run>/final-G.ckpt` on Drive. Module 3 (notebook 07)
is optional.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Free Drive space: the training notebooks save checkpoints and caches here (several GB in total).
# Deleted or overwritten files stay in the Drive Trash and still count -- empty it if space is low.
import shutil as _shutil
try:
    _du = _shutil.disk_usage('/content/drive/MyDrive')
    print(f'Google Drive: {_du.free / 1e9:.1f} GB free of {_du.total / 1e9:.1f} GB')
    if _du.free < 10e9:
        print('\u26a0 Under 10 GB free on Google Drive -- model files may fail to save. Empty the Trash '
              '(drive.google.com/drive/trash) or free space before training.')
except OSError:
    pass

import getpass, os, glob, json
GITHUB_TOKEN = getpass.getpass('GitHub Personal Access Token (repo scope): ')

REPO_OWNER = 'Mieka068'
REPO_NAME = 'DRProgression'
REPO_BRANCH = 'AllModuleEdits-PreFinals'  # <-- pre-finals fixes (CODE_FIXES_AllModuleEdits-PreFinals.md); 'mod-2-NonLongIntegration' = the earlier runs
REPO_CODE_SUBDIR = 'M2-DRProgression-VerM-module1-fgadr-poc'

DRIVE_DATA_DIR = '/content/drive/MyDrive/Thesis_Datasets'
assert os.path.isdir(DRIVE_DATA_DIR), f"Not found: {DRIVE_DATA_DIR}"
MODULE2_RUNS_DIR = os.path.join(DRIVE_DATA_DIR, 'module2_runs')
os.makedirs(MODULE2_RUNS_DIR, exist_ok=True)

# Which Module 1 models the pipeline uses -- set the SAME value in notebooks 04_05 and 06-10:
#   'drgnet' -- notebook 11's DRG-Net replication checkpoints (module1_runs/drgnet_replication/)
#   'poc'    -- notebooks 02/03's original POC checkpoints (module1_runs/)
# Everything built from Module 1's outputs (Module 1 caches, Module 2 runs, figures) gets a
# '_drgnet' suffix for 'drgnet', so the two pipelines never overwrite each other's results.
MODULE1_SOURCE = 'drgnet_combined'
assert MODULE1_SOURCE in ('drgnet_combined', 'drgnet', 'poc'), MODULE1_SOURCE
# New Drive folders for this branch's runs: every Module 1 cache / Module 2 / Module 3 folder below
# gets RUN_SUFFIX, so the mod-2-NonLongIntegration results stay untouched for before/after
# comparison. '' = read/write the earlier folders again.
RUN_SUFFIX = '_prefinals'
RUN_TAG = {'drgnet_combined': '_drgnet_comb', 'drgnet': '_drgnet', 'poc': ''}[MODULE1_SOURCE] + RUN_SUFFIX
# Module 2 resolution -- the SAME value in notebooks 06, 08, 09 and 10 (item 4.3; the scripts stop
# with an error on a mismatch). Go to 256 only once 128 px training works: both Module 2 stages must
# be retrained (~4x time per epoch) and the registration cache is rebuilt at that size.
MODULE2_IMAGE_SIZE = 128
MODULE1_RUNS = os.path.join(DRIVE_DATA_DIR, 'module1_runs', *(['drgnet_replication'] if MODULE1_SOURCE != 'poc' else []))
# Notebook 11 on this branch writes its models to drgnet_replication<RUN_SUFFIX>/ (tiled evaluation,
# 200 segmentation epochs, optional EyePACS grader); use those once they exist.
if MODULE1_SOURCE != 'poc' and os.path.isdir(os.path.join(MODULE1_RUNS + RUN_SUFFIX, 'segmentation')):
    MODULE1_RUNS += RUN_SUFFIX
elif MODULE1_SOURCE != 'poc' and RUN_SUFFIX:
    print(f'\u26a0 {MODULE1_RUNS + RUN_SUFFIX} not found -- using the earlier Module 1 models in {MODULE1_RUNS}. '
          'Run notebook 11 on this branch first for the re-trained ones.')
# Lesion segmenters: <MODULE1_RUNS>/segmentation/<SEG_MODEL_PREFIX><lesion>/model_2.pth.tar --
# FGADR + DDR combined (notebook 11) or FGADR only. The classifier is FGADR's either way.
SEG_MODEL_PREFIX = 'comb_models_' if MODULE1_SOURCE == 'drgnet_combined' else 'models_FGADR_NO_TATL_'
MODULE1_CACHE_DIR = os.path.join(DRIVE_DATA_DIR, 'module1_cache' + RUN_TAG)
os.makedirs(MODULE1_CACHE_DIR, exist_ok=True)
print('Module 1 source:', MODULE1_SOURCE, '| models:', MODULE1_RUNS, '| caches:', MODULE1_CACHE_DIR)

In [ ]:
# Clone this repo. Skips cleanly if already cloned in this runtime.
%cd /content
if not os.path.isdir(f'/content/{REPO_NAME}'):
    !git clone --branch {REPO_BRANCH} https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git {REPO_NAME}
REPO_CODE_DIR = f'/content/{REPO_NAME}/{REPO_CODE_SUBDIR}'
assert os.path.isdir(REPO_CODE_DIR), f"Not found: {REPO_CODE_DIR} -- check REPO_BRANCH/REPO_CODE_SUBDIR above"
%cd {REPO_CODE_DIR}
!pip install -q pandas openpyxl segmentation-models-pytorch matplotlib scipy torchmetrics torch-fidelity

In [ ]:
CLASSIFIER_RUN = 'fgadr'   # Module 1 grader: 'fgadr' (default) or 'eyepacs' (once an EyePACS grader is trained, notebook 11/09)
if not glob.glob(os.path.join(MODULE1_RUNS, 'classification', CLASSIFIER_RUN, 'saves', '*.pt')):
    print(f'\u26a0 No {CLASSIFIER_RUN} grader in {MODULE1_RUNS} yet -- using the FGADR one')
    CLASSIFIER_RUN = 'fgadr'

# Locate the Module 1 classifier + segmentation checkpoints (same discovery logic as
# notebooks 04-08). HE/SE are included automatically once they exist on Drive.
SEG_DIR   = os.path.join(MODULE1_RUNS, 'segmentation')
CLS_SAVES = os.path.join(MODULE1_RUNS, 'classification', CLASSIFIER_RUN, 'saves')
# Classifier input normalisation must match its training config (eyepacs_poc.yaml differs).
CLS_NORM_ARGS = ' --classifier-norm eyepacs' if CLASSIFIER_RUN == 'eyepacs' else ''

_pref  = ['final_weights.pt', 'best_validation_weights.pt']
_cands = [os.path.join(CLS_SAVES, n) for n in _pref] + sorted(
    glob.glob(os.path.join(CLS_SAVES, '*.pt')), key=os.path.getmtime, reverse=True)
CLS_CKPT = next((p for p in _cands if os.path.isfile(p)), None)
assert CLS_CKPT, f"No classifier .pt in {CLS_SAVES} -- run notebook 02/03 (MODULE1_SOURCE='poc') or 11 ('drgnet' / 'drgnet_combined') first"

SEG_ARGS = ''
for _lesion in ['EX', 'MA', 'HE', 'SE']:
    _p = os.path.join(SEG_DIR, f'{SEG_MODEL_PREFIX}{_lesion.lower()}', 'model_2.pth.tar')
    if os.path.isfile(_p):
        SEG_ARGS += f' --seg-checkpoint {_lesion}="{_p}"'
    print(f'{_lesion} seg :', _p if os.path.isfile(_p) else 'not trained yet')
assert 'EX=' in SEG_ARGS and 'MA=' in SEG_ARGS, f'EX/MA segmentation checkpoints missing in {SEG_DIR} -- run notebook 03 (poc) or 11 (drgnet) first'
print('classifier:', CLS_CKPT)

# Module 2 Generator: first one found, preferring the EyePACS-pretrained + fine-tuned run, and
# within a run best-G.ckpt (best epoch on held-out pairs) over final-G.ckpt (last epoch).
_gen_cands = [os.path.join(base, run + RUN_TAG, ckpt)
              for run in ('models_poc_tianjin_from_nonlong', 'models_poc_tianjin')
              for base in (MODULE2_RUNS_DIR, f'{REPO_CODE_DIR}/DRForestGAN-v2/stargan')
              for ckpt in ('best-G.ckpt', 'final-G.ckpt')]
GENERATOR_CKPT = next((p for p in _gen_cands if os.path.isfile(p)), None)
assert GENERATOR_CKPT, 'No Module 2 final-G.ckpt found -- run notebook 06 first (same MODULE1_SOURCE)'
print('generator :', GENERATOR_CKPT)

# Module 3 (optional): notebook 07's run, copied to Drive (module3_runs/module3_runs_poc{RUN_TAG}/), else this runtime.
MODULE3_DIR = next((d for d in (os.path.join(DRIVE_DATA_DIR, 'module3_runs', 'module3_runs_poc' + RUN_TAG),
                                 f'{REPO_CODE_DIR}/module3/module3_runs_poc')
                    if os.path.isfile(os.path.join(d, 'final-model.ckpt'))), None)
MODULE3_ARGS = ''
if MODULE3_DIR:
    _m3_ckpt = 'best-model.ckpt' if os.path.isfile(os.path.join(MODULE3_DIR, 'best-model.ckpt')) else 'final-model.ckpt'
    MODULE3_ARGS = (f' --module3-checkpoint "{MODULE3_DIR}/{_m3_ckpt}"'
                    f' --module3-thresholds-json "{MODULE3_DIR}/poc_results.json"')
print('module 3  :', MODULE3_ARGS or 'not found -- figures without progression-probability captions')

In [ ]:
# Input: a Drive folder of fundus images, or upload some now if that folder is empty/missing.
INPUT_DIR = os.path.join(DRIVE_DATA_DIR, 'progression_inputs')
os.makedirs(INPUT_DIR, exist_ok=True)
if not glob.glob(os.path.join(INPUT_DIR, '*')):
    from google.colab import files
    for _name, _data in files.upload().items():
        open(os.path.join(INPUT_DIR, _name), 'wb').write(_data)
print(sorted(os.listdir(INPUT_DIR)))

In [ ]:
START_STAGE = None   # None = use Module 1's grade of the input; or force a known clinical grade 0-3
END_STAGE = 4        # 4 = PDR
OUT_DIR = os.path.join(DRIVE_DATA_DIR, 'progression_outputs' + RUN_TAG)

_start = f' --start-stage {START_STAGE}' if START_STAGE is not None else ''
%cd {REPO_CODE_DIR}
!python generate_progression.py --image-size {MODULE2_IMAGE_SIZE} --input "{INPUT_DIR}" \
    --generator-checkpoint "{GENERATOR_CKPT}" \
    --classifier-checkpoint "{CLS_CKPT}"{SEG_ARGS}{CLS_NORM_ARGS}{MODULE3_ARGS} \
    --end-stage {END_STAGE}{_start} --out-dir "{OUT_DIR}"

In [ ]:
from IPython.display import Image as IPImage, display
for _fig in sorted(glob.glob(f'{OUT_DIR}/*/trajectory.png')):
    _summary = json.load(open(_fig.replace('trajectory.png', 'trajectory.json')))
    print(f"\n{_summary['input']}: Module 1 grade {_summary['module1_input_grade']}, start stage {_summary['start_stage']}")
    for _s in _summary['steps']:
        print(f"  stage {_s['stage']} ({_s['stage_name']}): LBS {_s['lbs']:.4f}"
              + (f", Module 1 re-grade {_s['module1_grade']} ({'consistent' if _s['module1_consistent'] else 'NOT consistent'})" if _s['synthesized'] else ' [input]'))
    display(IPImage(_fig))

## Caveats

- Module 1 does not segment neovascularisation (NV), the defining sign of PDR, so stage-4
  images mostly show heavier EX/HE/MA/SE unless the Generator learned NV from EyePACS stage-4
  images during pretraining (notebook 09).
- Each step builds on the previous synthesized image, so errors compound -- the PDR panel is
  the least reliable.
- "Module 1 re-grade ... NOT consistent" can mean the synthesized image is implausible, or that
  Module 1 is unreliable on synthesized images, or both.
- Module 3 captions on synthesized panels are outside its training distribution (untested).

## Before closing the runtime

Files written to Drive upload in the background. Run this last cell and wait for the ✓ before
disconnecting or deleting the runtime -- otherwise large files written at the end (model weights,
caches) can be lost.

In [ ]:
from google.colab import drive
drive.flush_and_unmount()
print('✓ Everything is uploaded to Google Drive -- safe to disconnect the runtime.')